# Generalized Estimating Equations (GEE) Examples

This notebook demonstrates GEE functionality in the RobustInfer Scala library.


In [1]:
%AddJar file:/app/scala_lib/build/libs/robustInfer-scala-spark34-0.1.0.jar

Starting download from file:/app/scala_lib/build/libs/robustInfer-scala-spark34-0.1.0.jar
Finished download of robustInfer-scala-spark34-0.1.0.jar
Using cached version of robustInfer-scala-spark34-0.1.0.jar


In [2]:
import robustinfer._
import org.apache.spark.sql.{Dataset, SparkSession}
import scala.util.Random


In [3]:
spark.version

3.4.1

## Generate Clustered Data

Let's create a simple clustered dataset for GEE analysis.


In [4]:
// Generate simple clustered data
val rng = new Random(42)
val numClusters = 50
val obsPerCluster = 4

val data = (1 to numClusters).flatMap { clusterId =>
  (1 to obsPerCluster).map { _ =>
    val x = rng.nextGaussian()
    val y = 1.0 + 0.5 * x + rng.nextGaussian() * 0.5  // Linear relationship with noise
    Obs(clusterId.toString, Array(x), y)
  }
}.toVector

val df = spark.createDataset(data)
println(f"Generated $numClusters clusters with $obsPerCluster observations each")
df.show(10)


Generated 50 clusters with 4 observations each
+---+--------------------+--------------------+---------+----+
|  i|                   x|                   y|timeIndex|   z|
+---+--------------------+--------------------+---------+----+
|  1|[1.1419053154730547]|   2.030656632227921|     null|null|
|  1|[-0.9498666368908...|-0.02842846164511681|     null|null|
|  1|[0.2809776380727795]|  1.4828002168527175|     null|null|
|  1|[-0.8172214073987...|-0.10693240503838508|     null|null|
|  2|[-0.1909445130708...|  1.6476344396598877|     null|null|
|  2|[0.8023071496873626]|  1.3403971125109346|     null|null|
|  2|[1.4105062239438624]|  1.3851367208651442|     null|null|
|  2|[-1.2096444592532...|  0.5720566193093687|     null|null|
|  3|[-0.4903496491990...|  1.0301859445376778|     null|null|
|  3|[-1.2035510019650...|   0.558732539338279|     null|null|
+---+--------------------+--------------------+---------+----+
only showing top 10 rows



rng = scala.util.Random@38eaff5f
numClusters = 50
obsPerCluster = 4
data = Vector(Obs(1,[D@5df3d169,2.030656632227921,None,None), Obs(1,[D@2c69a454,-0.02842846164511681,None,None), Obs(1,[D@4396b681,1.4828002168527175,None,None), Obs(1,[D@6977ee18,-0.10693240503838508,None,None), Obs(2,[D@2a4c12c6,1.6476344396598877,None,None), Obs(2,[D@4b6648df,1.3403971125109346,None,None), Obs(2,[D@3d32cd8c,1.3851367208651442,None,None), Obs(2,[D@6b63f31c,0.5720566193093687,None,None), Obs(3,[D@21731694,1.0301859445376778,None,None), Obs(3,[D@58aa82d5,0.558732539338279,None,None), Obs(3,[D@73df6ae1,1.9948389625571443,None,None), Obs(3,[D@7d09c95,2.0006145698963036,None,None), Obs(4,[D@3eaeed4d,0.8196146461589059,None,None), Obs(4,[D@329e6c67,1.465...


Vector(Obs(1,[D@5df3d169,2.030656632227921,None,None), Obs(1,[D@2c69a454,-0.02842846164511681,None,None), Obs(1,[D@4396b681,1.4828002168527175,None,None), Obs(1,[D@6977ee18,-0.10693240503838508,None,None), Obs(2,[D@2a4c12c6,1.6476344396598877,None,None), Obs(2,[D@4b6648df,1.3403971125109346,None,None), Obs(2,[D@3d32cd8c,1.3851367208651442,None,None), Obs(2,[D@6b63f31c,0.5720566193093687,None,None), Obs(3,[D@21731694,1.0301859445376778,None,None), Obs(3,[D@58aa82d5,0.558732539338279,None,None), Obs(3,[D@73df6ae1,1.9948389625571443,None,None), Obs(3,[D@7d09c95,2.0006145698963036,None,None), Obs(4,[D@3eaeed4d,0.8196146461589059,None,None), Obs(4,[D@329e6c67,1.465...

## Fit GEE Models

Let's fit GEE models with different correlation structures.


In [5]:
// GEE with Independent correlation structure
val geeIndependent = new GEE(Independent, Gaussian)
geeIndependent.fit(df, maxIter = 20, tol = 1e-6, verbose = true)
println("GEE with Independent correlation fitted")
geeIndependent.summary().show()


Iteration: DenseVector(1.0108532440068627, 0.4857544792235513), ||delta|| = 1.121508669161742
Iteration: DenseVector(1.0108532440068627, 0.4857544792235509), ||delta|| = 4.140725223971706E-16
Main iterations completed: 2, converged: true
GEE with Independent correlation fitted


geeIndependent = robustinfer.GEE@4c8d8e86


+---------+------------------+--------------------+------------------+-------+
|parameter|          estimate|           std_error|           z_score|p_value|
+---------+------------------+--------------------+------------------+-------+
|intercept|1.0108532440068627|0.038661142869995856|26.146491514904643|    0.0|
|    beta1|0.4857544792235509| 0.03865198282334474|12.567388365136303|    0.0|
+---------+------------------+--------------------+------------------+-------+



robustinfer.GEE@4c8d8e86

In [6]:
// GEE with Exchangeable correlation structure
val geeExchangeable = new GEE(Exchangeable, Gaussian)
geeExchangeable.fit(df, maxIter = 20, tol = 1e-6, verbose = true)
println("GEE with Exchangeable correlation fitted")
geeExchangeable.summary().show()


Updating R at warm-up iteration 0
Updated beta = DenseVector(1.009931031680866, 0.4921535408470315) at warm-up iteration 0
Updated beta = DenseVector(1.0099310316808656, 0.4921535408470313) at warm-up iteration 1
Warm-up iterations completed: 2, converged: true
Iteration: DenseVector(1.0108269665862186, 0.48593681340979833), ||delta|| = 0.006280955292268429
Iteration: DenseVector(1.0108269665862186, 0.48593681340979833), ||delta|| = 5.0443751154059466E-17
Main iterations completed: 2, converged: true
GEE with Exchangeable correlation fitted
+---------+-------------------+-------------------+------------------+-------+
|parameter|           estimate|          std_error|           z_score|p_value|
+---------+-------------------+-------------------+------------------+-------+
|intercept| 1.0108269665862186|0.03868055517147724|26.132690239451243|    0.0|
|    beta1|0.48593681340979833|0.03867364888317519|12.565062450603236|    0.0|
+---------+-------------------+-------------------+-------

geeExchangeable = robustinfer.GEE@44ba9e31


robustinfer.GEE@44ba9e31

In [7]:
geeExchangeable.correlationSummary().show()

+----------+--------------------+--------------------+--------------------+--------------------+
|time_point|              time_0|              time_1|              time_2|              time_3|
+----------+--------------------+--------------------+--------------------+--------------------+
|         0|                 1.0|0.008902511449414445|0.008902511449414445|0.008902511449414445|
|         1|0.008902511449414445|                 1.0|0.008902511449414445|0.008902511449414445|
|         2|0.008902511449414445|0.008902511449414445|                 1.0|0.008902511449414445|
|         3|0.008902511449414445|0.008902511449414445|0.008902511449414445|                 1.0|
+----------+--------------------+--------------------+--------------------+--------------------+



# Generate non-Gaussian data

In [8]:
// Generate Negative Binomial data
import breeze.linalg.DenseVector
import org.apache.spark.sql.SparkSession
import scala.util.Random

import breeze.stats.distributions.RandBasis
import org.apache.commons.math3.random.MersenneTwister
import breeze.stats.distributions.ThreadLocalRandomGenerator

implicit val randBasis: RandBasis = new RandBasis(new ThreadLocalRandomGenerator(new MersenneTwister(789)))

val rand = new Random(246)
val trueBeta = DenseVector(1.0, 0.5, -0.5)
val trueKappa = 0.2  // Overdispersion parameter
val nClusters = 1000
val obsPerCluster = 3

val data = (0 until nClusters).flatMap { clusterId =>
  val clusterEffect = rand.nextGaussian() * 0.2
  (0 until obsPerCluster).map { _ =>
    val x = Array(1.0, rand.nextGaussian(), rand.nextGaussian())
    val eta = x.zipWithIndex.map { case (xi, j) => xi * trueBeta(j) }.sum + clusterEffect
    val mu = math.exp(eta)
    // Generate NB using Gamma-Poisson mixture
    // If Z ~ Gamma(1/kappa, mu*kappa), then Y|Z ~ Poisson(Z) gives Y ~ NB
    val scale = trueKappa * mu
    val shape = 1.0 / trueKappa
    val gamma = breeze.stats.distributions.Gamma(shape, scale).draw()
    val y = breeze.stats.distributions.Poisson(gamma).draw().toDouble
    Obs(clusterId.toString, x.drop(1), y)
  }
}
val df = spark.createDataset(data)
df.show(10)


+---+--------------------+---+---------+----+
|  i|                   x|  y|timeIndex|   z|
+---+--------------------+---+---------+----+
|  0|[-0.4111475286603...|6.0|     null|null|
|  0|[1.35884923725915...|9.0|     null|null|
|  0|[-1.9759825393899...|3.0|     null|null|
|  1|[0.20872041183674...|4.0|     null|null|
|  1|[-0.5542727727572...|1.0|     null|null|
|  1|[-0.2991410411050...|0.0|     null|null|
|  2|[-1.2559549246792...|1.0|     null|null|
|  2|[1.04974578736168...|5.0|     null|null|
|  2|[-0.3419049118124...|4.0|     null|null|
|  3|[1.66077205628854...|6.0|     null|null|
+---+--------------------+---+---------+----+
only showing top 10 rows



randBasis = breeze.stats.distributions.RandBasis@282d58cc
rand = scala.util.Random@70f283b0
trueBeta = DenseVector(1.0, 0.5, -0.5)
trueKappa = 0.2
nClusters = 1000
obsPerCluster = 3
data = Vector(Obs(0,[D@1916a882,6.0,None,None), Obs(0,[D@34f098fe,9.0,None,None), Obs(0,[D@76d95d94,3.0,None,None), Obs(1,[D@12d2bffc,4.0,None,None), Obs(1,[D@3e2c479c,1.0,None,None), Obs(1,...


Vector(Obs(0,[D@1916a882,6.0,None,None), Obs(0,[D@34f098fe,9.0,None,None), Obs(0,[D@76d95d94,3.0,None,None), Obs(1,[D@12d2bffc,4.0,None,None), Obs(1,[D@3e2c479c,1.0,None,None), Obs(1,...

In [9]:
val gee = new GEE(Exchangeable,family = NegativeBinomial)
gee.fit(df, warmupStepSize=10, warmupRounds=5, maxIter = 50, tol = 1e-5,verbose = true)
gee.summary().show()

Updating R at warm-up iteration 0
Updated kappa = 24.176333333333332 at warm-up iteration 0
Updated beta = DenseVector(2.572146988651835, 1.9099993879965211, -1.923672744569841) at warm-up iteration 0
Updated beta = DenseVector(2.3974805580751335, 1.0666447827302794, -1.034091688226813) at warm-up iteration 1
Updated beta = DenseVector(1.7179489251457167, 0.9093021954747207, -0.882769440668836) at warm-up iteration 2
Updated beta = DenseVector(1.2748502671458766, 0.7122981781367264, -0.6949849575197349) at warm-up iteration 3
Updated beta = DenseVector(1.0613106737582991, 0.5764322103383297, -0.5657605551177778) at warm-up iteration 4
Updated beta = DenseVector(1.0078798056790954, 0.5360776699559249, -0.5270148443740534) at warm-up iteration 5
Updated beta = DenseVector(1.0043584101353515, 0.5327676388998147, -0.5237829475840459) at warm-up iteration 6
Updated beta = DenseVector(1.0043159032470175, 0.5326806559572211, -0.5236993395343738) at warm-up iteration 7
Updated beta = DenseVect

gee = robustinfer.GEE@18548606


robustinfer.GEE@18548606

In [10]:
gee.correlationSummary().show()

+----------+-------------------+-------------------+-------------------+
|time_point|             time_0|             time_1|             time_2|
+----------+-------------------+-------------------+-------------------+
|         0|                1.0|0.03457423834003138|0.03457423834003138|
|         1|0.03457423834003138|                1.0|0.03457423834003138|
|         2|0.03457423834003138|0.03457423834003138|                1.0|
+----------+-------------------+-------------------+-------------------+

